In [ ]:
import random
from datasets import load_dataset, Dataset
import os
import time

# =================================================================================
# 🧠 AI 코딩 튜터에게 문의합니다! 🧙‍♂️
# 데이터셋 이름: nayohan/Magpie-Gemma2-Pro-200K-Filtered-ko
# 데이터셋 의미: 대규모 한국어 지시어-응답 쌍으로 구성된 LLM 학습용 데이터셋입니다.
# 데이터셋 설명: 사용자가 주어진 명령어(instruction)에 따라 AI가 적절하게 응답(response)하는 능력을 학습시키는 데 사용됩니다.
# 즉, '어떻게 질문해야 하는지'와 '어떻게 대답해야 하는지'를 체계적으로 학습한 보물창고와 같아요!
# =================================================================================

# 사용 상수 정의
DATASET_NAME = "nayohan/Magpie-Gemma2-Pro-200K-Filtered-ko"
SAMPLE_COUNT = 5  # 🔥 초보자를 위해 단 5개의 샘플만 살펴보겠습니다! 너무 적어서 귀엽죠?

def load_magpie_dataset():
    """
    데이터셋을 로드합니다. 스트리밍 모드를 먼저 시도하고, 실패할 경우 일반 모드로 전환합니다.
    (AI 튜터: 데이터 로딩은 언제나 가장 까다로운 관문입니다! 하지만 걱정 마세요, 제가 완벽하게 로드해 드릴게요!)
    """
    print(f"🔍 데이터셋 '{DATASET_NAME}' 로드를 시도합니다...")
    
    dataset = None
    
    # 1. 스트리밍 모드(Streaming)를 먼저 시도합니다. (대용량 데이터를 메모리 걱정 없이 천천히 보기 위함)
    try:
        print("✅ 스트리밍 모드 (streaming=True)로 로딩을 시도합니다...")
        # 반드시 split을 명시해야 합니다!
        dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
        print("🌟 [성공] 스트리밍 모드 로드 성공! 데이터가 필요할 때마다 가져올 수 있습니다.")
        return dataset
    
    except Exception as e:
        # 2. 만약 스트리밍 모드가 실패하거나 환경 문제로 인해 오류가 발생하면 (예: 메모리 부족), 일반 모드로 전환합니다.
        print(f"\n⚠️ [주의] 스트리밍 로드 실패: {e.__class__.__name__} 오류 발생.")
        print("🔄 일반 모드 (Dataset)로 전환하여 소규모 데이터만 다운로드합니다.")
        try:
            # 'test' 분할이 없는 경우, 그냥 'train'으로 소량 다운로드 시도
            dataset = load_dataset(DATASET_NAME, split='train')
            return dataset
        except Exception as e_fallback:
            print(f"❌ 데이터셋 로드에 실패했습니다. {e_fallback} 확인 필요.")
            return None

def analyze_sample_data(dataset_iterator):
    """
    로드된 데이터셋의 샘플들을 순회하며, 데이터가 어떤 내용을 담고 있는지 분석하는 함수입니다.
    """
    print("\n" + "="*60)
    print("📚 1단계: 데이터셋의 핵심 열(Features) 탐색하기")
    print("="*60)
    print("✨ (튜터 코멘트) 이 데이터셋은 '질문(Instruction)'과 '답변(Response)'이 핵심입니다. "
          "다른 필드들은 학습 과정에서 사용된 '꼬리표' 같은 것이라 재미있게 보는 건 다음에 해요!")
    
    sample_data_list = []
    
    # 스트리밍 혹은 일반 데이터셋에서 상위 K개만 샘플링하여 리스트로 변환 (최대 5개)
    # Constraint 9, 16 적용: .take() 후 list() 변환
    try:
        sample_data_list = list(dataset_iterator.take(SAMPLE_COUNT))
    except AttributeError:
        print("🚨 데이터셋 객체에 'take' 메서드가 없습니다. 로딩에 실패했을 수 있습니다.")
        return

    if not sample_data_list:
        print("😭 로드된 데이터가 없습니다. DATASET_NAME을 확인해주세요.")
        return

    print(f"\n✨ 성공! 상위 {min(SAMPLE_COUNT, len(sample_data_list))}개의 샘플을 불러와 열어봤어요.")
    
    for i, sample in enumerate(sample_data_list):
        print(f"\n--- 샘플 {i+1} 🧠 ---")
        
        instruction = sample.get('instruction', 'N/A')
        response = sample.get('response', 'N/A')
        intent = sample.get('intent', 'N/A')
        task_category = sample.get('task_category', 'N/A')
        
        print(f"💡 [지시(Instruction)] : {instruction[:40].replace('\n', ' ')}... (길이: {len(str(instruction))})")
        print(f"🤖 [응답(Response)]   : {response[:40].replace('\n', ' ')}... (길이: {len(str(response))})")
        print(f"🎯 [의도(Intent)]     : {intent}")
        print(f"🏷️ [분류(Task)]       : {task_category}")

def creative_ai_practice(dataset_iterator):
    """
    가장 창의적인 실습: 'Prompt 재해석 마법사' 되기! ✨
    주어진 질문(Instruction)과 답변(Response)을 바탕으로, 이 사용자가 어떤 의도(Intent)로 질문했는지 추론하고,
    더 나은 프롬프트 템플릿을 제안하는 시뮬레이션입니다.
    (튜터 코멘트: 단순히 데이터 읽기를 넘어, AI가 생각하는 과정을 경험해 봅시다!)
    """
    print("\n" + "="*80)
    print("🚀 2단계: 창의 AI 실습! '프롬프트 의도 재해석 마법사' 되기")
    print("="*80)
    print("🤔 (목표) 우리는 사용자가 작성한 '생각의 조각(Instruction)'을 받아, "
          "숨겨진 '진짜 질문 의도(Intent)'를 추론하고, 완벽한 '프롬프트 템플릿'을 만들어 낼 거예요.")

    print("\n✨ (참고) 이 과정은 실제 AI 모델의 추론 과정을 시뮬레이션한 것이에요.")
    
    # 샘플 데이터를 재활용
    sample_data_list = []
    try:
        sample_data_list = list(dataset_iterator.take(SAMPLE_COUNT))
    except AttributeError:
        print("🚨 샘플링에 실패했습니다. 전 단계를 다시 확인해주세요.")
        return

    for i, sample in enumerate(sample_data_list):
        instruction = sample.get('instruction', '')
        response = sample.get('response', '')
        intent = sample.get('intent', 'N/A')
        
        print(f"\n--- 🪄 {i+1}번째 샘플 분석 ---")
        print(f"   [원문 지시]: {instruction[:50].strip()}...")
        print(f"   [AI 응답]: {response[:50].strip()}...")
        print("-" * 30)

        # 1. Intent 추론 시뮬레이션 (데이터셋의 의도를 활용)
        print(f"✅ 추론된 사용자의 의도(Intent): '{intent}'")
        
        # 2. 프롬프트 개선 제안 (튜터의 위트 발휘!)
        if "요약" in intent or "summary" in intent:
            print("💡 [튜터 제안]: 이 사용자는 '정보 간결화'를 원하는 것 같습니다. 역할(Role)을 명시해주면 더 좋습니다!")
            print("   [추천 템플릿] : '당신은 전문가 요약가입니다. 다음 [텍스트]를 세 가지 핵심 포인트로 요약해주세요.'")
        elif "질문" in intent or "question" in intent:
            print("💡 [튜터 제안]: 사용자는 '지식 검색'을 원합니다. 반드시 어떤 배경지식(Context)을 참조해야 하는지 알려주세요!")
            print("   [추천 템플릿] : '다음 [배경지식]을 참고하여, [주제]에 대해 질문 형식으로 답해주세요.'")
        else:
            print("💡 [튜터 제안]: 범용적인 상호작용이 예상됩니다. 질문의 목적과 기대하는 답변의 형식을 구체적으로 정의해주는 것이 가장 좋습니다!")

# =================================================================================
# 🚀 메인 실행 로직
# =================================================================================

# 1. 데이터셋 로드 (스트리밍/폴백 로직 수행)
dataset_iterator = load_magpie_dataset()

if dataset_iterator:
    # 2. 데이터 구조 분석
    analyze_sample_data(dataset_iterator)
    
    # 3. 창의적 AI 실습
    creative_ai_practice(dataset_iterator)
else:
    print("\n😭 데이터셋 로드에 실패하여 실습을 진행할 수 없습니다. 오류 메시지를 확인해주세요!")